# 02b · Solucion: limpieza de datos sucios en Polars

> Solucion comentada del ejercicio. Todas las operaciones en Polars 2.x.

In [ ]:
import polars as pl
from pathlib import Path

CSV = Path("datos_sucios.csv")
df = pl.read_csv(CSV)
print(f"filas iniciales: {df.height}")

## 1. Variantes de ciudad

In [ ]:
conteo_ciudad = df.group_by("ciudad").len().sort("len", descending=True)
conteo_ciudad

Encontramos variantes con/sin tildes y mayusculas/minusculas por cada ciudad canonica.

## 2. Normalizar `ciudad`

In [ ]:
acentos = {"á": "a", "é": "e", "í": "i", "ó": "o", "ú": "u",
           "à": "a", "è": "e", "ì": "i", "ò": "o", "ù": "u",
           "ç": "c"}
expr = pl.col("ciudad").str.to_lowercase()
for k, v in acentos.items():
    expr = expr.str.replace_all(k, v)
df = df.with_columns(expr.str.to_titlecase().alias("ciudad_norm"))
df.group_by("ciudad_norm").len().sort("len", descending=True)

## 3. Fechas con dos formatos

In [ ]:
df = df.with_columns(
    pl.when(pl.col("fecha").str.contains("-"))
    .then(pl.col("fecha").str.strptime(pl.Date, format="%Y-%m-%d", strict=False))
    .otherwise(pl.col("fecha").str.strptime(pl.Date, format="%d/%m/%Y", strict=False))
    .alias("fecha_dt")
)
df.select("fecha", "fecha_dt").head()

## 4. Precio y moneda

In [ ]:
df = df.with_columns(
    pl.when(pl.col("precio").str.starts_with("USD:"))
    .then(pl.lit("USD"))
    .otherwise(pl.lit("EUR"))
    .alias("moneda"),
    pl.col("precio").str.replace_all("USD:", "").cast(pl.Float64).alias("precio_eur"),
)
df.select("precio", "precio_eur", "moneda").head()

## 5. Valores centinela en `cantidad`

In [ ]:
df = df.with_columns(
    pl.when(pl.col("cantidad") == -999).then(None).otherwise(pl.col("cantidad")).alias("cantidad")
)
print(f"nulos en cantidad: {df['cantidad'].null_count()}")

## 6. Email

In [ ]:
df = df.with_columns(
    pl.when((pl.col("email") == "unknown") | (pl.col("email") == ""))
    .then(None)
    .otherwise(pl.col("email"))
    .alias("email")
)
print(f"nulos en email: {df['email'].null_count()}")

## 7. Duplicados

In [ ]:
exactos = df.height - df.unique().height
cercanos = df.unique().height - df.unique(subset=["ciudad_norm", "fecha_dt", "precio_eur", "producto"]).height
print(f"duplicados exactos: {exactos}")
print(f"duplicados cercanos (misma venta, cantidad distinta): {cercanos}")

## Tabla final de hallazgos

| Problema | Cuantos registros | Decision |
|---|---|---|
| Variantes de ciudad | 5 ciudades × 4 variantes | Normalizar (minusculas sin tildes + titlecase). |
| Fechas en dos formatos | ~100 ISO + ~100 euro | Parsear ambas; dejar `pl.Date`. |
| Precios en USD | ~30 % | Separar en `moneda`; convertir si procede (fuera de alcance aqui). |
| `cantidad = -999` | ~5 % | Convertir a `null`; U3 imputara. |
| `email` "unknown" o vacio | ~13 % | Convertir a `null`. |
| Duplicados exactos | 3 | Eliminar. |
| Duplicados cercanos | 3 | Investigar — ¿devoluciones?; en este caso eliminar. |